## Exercise 25 - Generative Adversarial Network for MNIST Digits

Estimated time: **35-40 minutes**

The goal of this exercise is to become familiar with a relatively simple Generative Adversarial Network (GAN), which attempts to generate the MNIST digits.

The GAN model is created based on the official TensorFlow documentation and their Generative Adversarial Network (GAN) tutorial, which is a common and well-established approach for building such models.

The core concepts are from the paper "Generative Adversarial Nets" by Ian Goodfellow and other researchers, which introduced the GAN framework.

- The images are generated every 5 epochs and stored in the generated_images folder
- You can download the images to your computer and view them
- Try experimenting with different number of epochs (presently it is set at 50) to see if there is an improvement in image quality
- Execute this exercise on **T4 GPU**

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os

# --- 1. Hyperparameters and Constants ---
BUFFER_SIZE = 60000
BATCH_SIZE = 256
EPOCHS = 50
NOISE_DIM = 100
NUM_EXAMPLES_TO_GENERATE = 16

# Directory to save generated images
SAVE_DIR = 'generated_images'
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# --- 2. Data Loading and Preprocessing ---
def load_and_preprocess_data():
    """Loads the MNIST dataset and preprocesses it."""
    (train_images, train_labels), (_, _) = tf.keras.datasets.mnist.load_data()

    # Reshape and normalize images to be in the range [-1, 1]
    # This is a common practice for GANs
    train_images = train_images.reshape(train_images.shape[0], 28, 28, 1).astype('float32')
    train_images = (train_images - 127.5) / 127.5

    # Create a batched and shuffled dataset
    train_dataset = tf.data.Dataset.from_tensor_slices(train_images).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
    return train_dataset

# --- 3. Generator Model ---
def make_generator_model():
    """
    Creates a generator model.
    It takes a random noise vector as input and upsamples it
    to generate an image of size 28x28x1.
    """
    model = tf.keras.Sequential()

    # Input layer takes a noise vector and projects it into a small
    # but high-dimensional space.
    model.add(layers.Input(shape=(NOISE_DIM,)))
    model.add(layers.Dense(7*7*256, use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Reshape the output to a 4D tensor representing the starting image
    model.add(layers.Reshape((7, 7, 256)))
    assert model.output_shape == (None, 7, 7, 256) # Note: None is the batch size

    # Upsampling block 1:
    # Transposed convolution to upsample from 7x7 to 14x14
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    assert model.output_shape == (None, 7, 7, 128)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Upsampling block 2:
    # Transposed convolution to upsample from 14x14 to 28x28
    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 14, 14, 64)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Output layer:
    # Transposed convolution to generate the final 28x28x1 image
    # Tanh activation scales the output to [-1, 1], matching the input data
    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 28, 28, 1)

    return model

# --- 4. Discriminator Model ---
def make_discriminator_model():
    """
    Creates a discriminator model.
    It takes an image as input and classifies it as either real or fake.
    """
    model = tf.keras.Sequential()

    # Input layer: 28x28x1 image
    # Convolutional layer 1
    model.add(layers.Input(shape=[28, 28, 1]))
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    # Convolutional layer 2
    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    # Flatten the output to feed into a dense layer
    model.add(layers.Flatten())

    # Output layer: a single neuron with a sigmoid activation
    # This neuron outputs the probability that the image is real (1) or fake (0)
    model.add(layers.Dense(1, activation='sigmoid'))

    return model

# --- 5. Loss Functions and Optimizers ---
# Use Binary Cross-Entropy for the loss function
cross_entropy = tf.keras.losses.BinaryCrossentropy()

def discriminator_loss(real_output, fake_output):
    """Calculates the discriminator's loss."""
    # Loss for real images (should be close to 1)
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    # Loss for fake images (should be close to 0)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    """
    Calculates the generator's loss.
    The generator wants the discriminator to output 1s for its fakes.
    """
    return cross_entropy(tf.ones_like(fake_output), fake_output)

# Use Adam optimizers for both models
generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

# --- 6. Training Step ---
# Use a tf.function decorator for faster execution
@tf.function
def train_step(images):
    """
    Performs one training step for both the generator and discriminator.
    The training happens in two parts:
    1. Train the discriminator on real and fake images.
    2. Train the generator to fool the discriminator.
    """
    # Generate random noise for the generator
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # Generate images with the generator
        generated_images = generator(noise, training=True)

        # Get discriminator's predictions for both real and generated images
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        # Calculate losses for both models
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    # Calculate gradients
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    # Apply gradients to update the models' weights
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

# --- 7. Visualization Function ---
def save_images(model, epoch, test_input):
    """Saves a plot of generated images."""
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        # Rescale the image from [-1, 1] to [0, 1] for plotting
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')
    plt.savefig(os.path.join(SAVE_DIR, f'image_at_epoch_{epoch:04d}.png'))
    plt.close()

# --- 8. Main Training Loop ---
def train(dataset, epochs):
    """
    The main training loop.
    It iterates over the dataset for a given number of epochs.
    """
    # Create a seed to visualize the progress of the generator
    seed = tf.random.normal([NUM_EXAMPLES_TO_GENERATE, NOISE_DIM])

    for epoch in range(epochs):
        # Iterate over the dataset in batches
        for image_batch in dataset:
            train_step(image_batch)

        # Save generated images every few epochs
        if (epoch + 1) % 5 == 0:
            save_images(generator, epoch + 1, seed)
            print(f'Epoch {epoch + 1} completed.')

    # Save a final image
    save_images(generator, epochs, seed)
    print("Training finished.")

# --- 9. Run the Program ---
if __name__ == '__main__':
    print("Loading and preparing data...")
    train_dataset = load_and_preprocess_data()
    print("Data loaded.")

    print("Creating generator and discriminator models...")
    generator = make_generator_model()
    discriminator = make_discriminator_model()
    print("Models created.")

    print("Starting training...")
    train(train_dataset, EPOCHS)

    # After training, you can visualize the final generator output
    print(f"Generated images are saved in the '{SAVE_DIR}' folder.")
    final_noise = tf.random.normal([1, NOISE_DIM])
    final_image = generator(final_noise, training=False)
